# CartIQ — Phase 7: Model Comparison & Error Analysis
**CS Machine Learning Course Project — Group 3**

Unified comparison of all four models, cold-start analysis, and error characterization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 7.1 Unified Model Comparison Table

In [ ]:
# Load all metrics
trad = pd.read_csv('models/metrics_traditional.csv')
ncf = pd.read_csv('models/metrics_ncf.csv')
gru = pd.read_csv('models/metrics_gru.csv')

# Combine
comparison = pd.concat([trad, ncf, gru], ignore_index=True)
comparison = comparison.set_index('model')

# Format for display
display_cols = ['precision', 'recall', 'f1', 'roc_auc']
styled = comparison[display_cols].style.format('{:.4f}').highlight_max(axis=0, color='#bbf7d0')

print('═══════════════════════════════════════════════════════════')
print('  CARTIQ — FINAL MODEL COMPARISON')
print('═══════════════════════════════════════════════════════════')
display(styled)

In [ ]:
# Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

metrics_to_plot = ['precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metrics_to_plot))
width = 0.18

colors = ['#3b82f6', '#ef4444', '#10b981', '#f59e0b']
models = comparison.index.tolist()

for i, model in enumerate(models):
    values = [comparison.loc[model, m] for m in metrics_to_plot]
    offset = (i - len(models)/2 + 0.5) * width
    bars = ax.bar(x + offset, values, width, label=model, color=colors[i], edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(['Precision', 'Recall', 'F1-Score', 'ROC-AUC'], fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('CartIQ — Model Performance Comparison', fontweight='bold', fontsize=16)
ax.legend(loc='upper right', fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.2 Radar Chart

In [ ]:
from matplotlib.patches import Patch

categories = ['Precision', 'Recall', 'F1', 'ROC-AUC']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, model in enumerate(models):
    values = [comparison.loc[model, m] for m in metrics_to_plot]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Model Performance Radar', fontweight='bold', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig('model_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.3 Cold-Start Analysis
Performance on users with fewer than 5 orders vs. experienced users.

In [ ]:
# Reload test data with user info
test_df = pd.read_parquet('data/test.parquet')
feature_cols = pd.read_csv('data/feature_cols.csv', header=None)[0].tolist()

# Cold-start threshold
COLD_THRESHOLD = 5

cold_users = test_df[test_df['u_total_orders'] < COLD_THRESHOLD]['user_id'].unique()
warm_users = test_df[test_df['u_total_orders'] >= COLD_THRESHOLD]['user_id'].unique()

cold_df = test_df[test_df['user_id'].isin(cold_users)]
warm_df = test_df[test_df['user_id'].isin(warm_users)]

print(f'Cold-start users (<{COLD_THRESHOLD} orders): {len(cold_users):,} users, {len(cold_df):,} rows')
print(f'Warm users (>={COLD_THRESHOLD} orders):      {len(warm_users):,} users, {len(warm_df):,} rows')

In [ ]:
import pickle
import lightgbm as lgb

# Load RF and GBDT
with open('models/random_forest.pkl', 'rb') as f:
    rf = pickle.load(f)
gbdt = lgb.Booster(model_file='models/lightgbm.txt')

# Evaluate on cold vs warm
results = []

for subset_name, subset_df in [('Cold-Start', cold_df), ('Warm', warm_df)]:
    X = subset_df[feature_cols].values
    y = subset_df['label'].values
    
    if len(y) == 0 or y.sum() == 0:
        continue
    
    # RF
    rf_prob = rf.predict_proba(X)[:, 1]
    rf_p = (rf_prob >= 0.5).astype(int)
    results.append({
        'Subset': subset_name, 'Model': 'Random Forest',
        'F1': f1_score(y, rf_p), 'AUC': roc_auc_score(y, rf_prob),
    })
    
    # GBDT
    gb_prob = gbdt.predict(X)
    gb_p = (gb_prob >= 0.5).astype(int)
    results.append({
        'Subset': subset_name, 'Model': 'LightGBM',
        'F1': f1_score(y, gb_p), 'AUC': roc_auc_score(y, gb_prob),
    })

cold_results = pd.DataFrame(results)
display(cold_results.pivot(index='Model', columns='Subset', values=['F1', 'AUC']))

In [ ]:
# Cold-start bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric in zip(axes, ['F1', 'AUC']):
    pivot = cold_results.pivot(index='Model', columns='Subset', values=metric)
    pivot.plot(kind='bar', ax=ax, color=['#ef4444', '#10b981'], edgecolor='white', rot=0)
    ax.set_title(f'{metric} — Cold-Start vs Warm Users', fontweight='bold')
    ax.set_ylabel(metric)
    ax.legend(title='User Group')

plt.tight_layout()
plt.savefig('cold_start_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.4 Error Analysis — False Positive / False Negative Characterization

In [ ]:
# Use GBDT (best model) for error analysis
X_test = test_df[feature_cols].values
y_test = test_df['label'].values
gbdt_proba = gbdt.predict(X_test)
gbdt_pred = (gbdt_proba >= 0.5).astype(int)

test_df = test_df.copy()
test_df['predicted'] = gbdt_pred
test_df['proba'] = gbdt_proba

fp = test_df[(test_df['label'] == 0) & (test_df['predicted'] == 1)]
fn = test_df[(test_df['label'] == 1) & (test_df['predicted'] == 0)]
tp = test_df[(test_df['label'] == 1) & (test_df['predicted'] == 1)]

print(f'True Positives:  {len(tp):,}')
print(f'False Positives: {len(fp):,}')
print(f'False Negatives: {len(fn):,}')
print(f'\n--- False Positives (predicted reorder but didn\'t) ---')
print(f'Avg up_times_ordered:    {fp["up_times_ordered"].mean():.1f}')
print(f'Avg up_reorder_ratio:    {fp["up_reorder_ratio"].mean():.3f}')
print(f'Avg u_total_orders:      {fp["u_total_orders"].mean():.1f}')
print(f'\n--- False Negatives (missed actual reorders) ---')
print(f'Avg up_times_ordered:    {fn["up_times_ordered"].mean():.1f}')
print(f'Avg up_reorder_ratio:    {fn["up_reorder_ratio"].mean():.3f}')
print(f'Avg u_total_orders:      {fn["u_total_orders"].mean():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in [
    (axes[0], 'up_times_ordered', 'Times Ordered'),
    (axes[1], 'up_reorder_ratio', 'Reorder Ratio'),
    (axes[2], 'u_total_orders', 'User Total Orders'),
]:
    for subset, label, color in [
        (tp, 'TP', '#10b981'),
        (fp, 'FP', '#f59e0b'),
        (fn, 'FN', '#ef4444'),
    ]:
        ax.hist(subset[col].clip(upper=subset[col].quantile(0.95)),
                bins=30, alpha=0.5, label=label, color=color, density=True)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_ylabel('Density')

plt.suptitle('Error Analysis — LightGBM (GBDT)', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.5 Final Summary

In [ ]:
print('╔══════════════════════════════════════════════════════════════╗')
print('║              CartIQ — FINAL RESULTS SUMMARY                ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║                                                            ║')
print('║  Model              Prec    Rec     F1      AUC            ║')
print('║  ─────────────────  ─────   ─────   ─────   ─────          ║')

for model in comparison.index:
    p = comparison.loc[model, 'precision']
    r = comparison.loc[model, 'recall']
    f = comparison.loc[model, 'f1']
    a = comparison.loc[model, 'roc_auc']
    marker = ' ◀ BEST' if f == comparison['f1'].max() else ''
    print(f'║  {model:19s} {p:.4f}  {r:.4f}  {f:.4f}  {a:.4f}{marker:>10s} ║')

print('║                                                            ║')
print('║  Key Findings:                                             ║')
print('║  • GBDT excels on structured tabular features              ║')
print('║  • NCF captures latent user-item interactions              ║')
print('║  • GRU captures sequential purchasing patterns             ║')
print('║  • Cold-start remains challenging across all models        ║')
print('║                                                            ║')
print('╚══════════════════════════════════════════════════════════════╝')

In [ ]:
# Save final comparison
comparison.to_csv('models/final_comparison.csv')
print('Final comparison saved to models/final_comparison.csv')
print('\nAll model artifacts in models/ directory:')
import os
for f in sorted(os.listdir('models')):
    size = os.path.getsize(os.path.join('models', f)) / 1e6
    print(f'  {f:35s} {size:8.4f} MB')